## Dataset Loading and preparation

In [32]:
# Section 1: Imports & Paths
import pandas as pd
import numpy as np
import os
import pickle
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, roc_curve
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
import shap
import optuna
import warnings
warnings.filterwarnings("ignore")

# Paths
DATA_PATH = "E:\Thesis\llm-ids-shield\Data\Processed\Preprocessed_file.csv"   # Change to your dataset path
RESULTS_DIR = r"E:\Thesis\llm-ids-shield\Notebooks\results"
SHAP_DIR = os.path.join(RESULTS_DIR, "shap_plots")
os.makedirs(SHAP_DIR, exist_ok=True)


In [33]:
# Section 2: Load Dataset
df = pd.read_csv(DATA_PATH)

# Encode labels if they are categorical
if df['Label'].dtype == 'object':
    le = LabelEncoder()
    df['Label'] = le.fit_transform(df['Label'])

# Separate features and target
X = df.drop('Label', axis=1)
y = df['Label']

print("Dataset Shape:", df.shape)
print("Feature Columns:", X.columns.tolist())


Dataset Shape: (625783, 23)
Feature Columns: ['Flow_Duration', 'Init_Bwd_Win_Byts', 'Dst_Port', 'Idle_Mean', 'Flow_IAT_Min', 'Flow_Pkts/s', 'Pkt_Len_Max', 'ACK_Flag_Cnt', 'Fwd_Pkt_Len_Max', 'Bwd_Header_Len', 'TotLen_Bwd_Pkts', 'Fwd_Pkts/s', 'Flow_Byts/s', 'Bwd_Pkt_Len_Max', 'Bwd_Pkts/s', 'TotLen_Fwd_Pkts', 'Flow_IAT_Max', 'Fwd_Pkt_Len_Min', 'Bwd_Pkt_Len_Mean', 'Pkt_Len_Std', 'SYN_Flag_Cnt', 'Pkt_Len_Mean']


In [34]:
# Section 3: Utility Functions

# 1. Metrics calculation
def compute_metrics(y_true, y_pred, y_prob=None, classes=None):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted')
    rec = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    cm = confusion_matrix(y_true, y_pred)

    # Only compute ROC AUC if y_prob is valid
    roc_auc = None
    if y_prob is not None:
        try:
            roc_auc = roc_auc_score(y_true, y_prob, multi_class='ovr')
        except ValueError:
            print("Skipping ROC AUC for this model/fold due to class mismatch.")
    
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "confusion_matrix": cm, "roc_auc": roc_auc}


# 2. Plot Confusion Matrix
def plot_confusion_matrix(cm, labels, title="Confusion Matrix"):
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=labels, yticklabels=labels, cmap="Blues")
    plt.title(title)
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.show()

# 3. Save model
def save_model(model, filename):
    with open(filename, "wb") as f:
        pickle.dump(model, f)

# 4. SHAP Explainability
def shap_explain(model, X, model_name="model"):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)
    shap.summary_plot(shap_values, X, show=False)
    plt.savefig(os.path.join(SHAP_DIR, f"{model_name}_shap_summary.png"))
    plt.close()


In [4]:
# Section 4: Optuna Tuning
def optuna_tuning_xgb(X, y, n_trials=30):
    def objective(trial):
        param = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "max_depth": trial.suggest_int("max_depth", 3, 15),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "gamma": trial.suggest_float("gamma", 0, 5)
        }
        kf = KFold(n_splits=3, shuffle=True, random_state=42)
        f1_scores = []
        for train_idx, val_idx in kf.split(X):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            model = XGBClassifier(**param, use_label_encoder=False, eval_metric='mlogloss')
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            f1_scores.append(f1_score(y_val, y_pred, average='weighted'))
        return np.mean(f1_scores)
    
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)
    return study.best_params

def optuna_tuning_lgb(X, y, n_trials=30):
    def objective(trial):
        param = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "max_depth": trial.suggest_int("max_depth", 3, 15),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
            "num_leaves": trial.suggest_int("num_leaves", 20, 150),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0)
        }
        kf = KFold(n_splits=3, shuffle=True, random_state=42)
        f1_scores = []
        for train_idx, val_idx in kf.split(X):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            model = LGBMClassifier(**param)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            f1_scores.append(f1_score(y_val, y_pred, average='weighted'))
        return np.mean(f1_scores)
    
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)
    return study.best_params



In [5]:
def optuna_tuning_rf(X, y, n_trials=30):
    def objective(trial):
        param = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "max_depth": trial.suggest_int("max_depth", 3, 30),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5)
        }
        kf = KFold(n_splits=3, shuffle=True, random_state=42)
        f1_scores = []
        for train_idx, val_idx in kf.split(X):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            model = RandomForestClassifier(**param,n_jobs=-1,)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            f1_scores.append(f1_score(y_val, y_pred, average='weighted'))
        return np.mean(f1_scores)
    
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)
    return study.best_params


In [35]:
# Section 5: k-Fold Training
def train_models_kfold(X, y, models_dict, k=5):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    all_metrics = {name: [] for name in models_dict.keys()}
    fold_num = 1

    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        for name, model in models_dict.items():
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            y_prob = model.predict_proba(X_val) if hasattr(model, "predict_proba") else None
            metrics = compute_metrics(y_val, y_pred, y_prob)
            metrics['fold'] = fold_num
            all_metrics[name].append(metrics)
        fold_num += 1
    return all_metrics


In [36]:
# json serializable conversion
import numpy as np
import json

def make_json_serializable(metrics_all):
    serializable = {}
    for model, folds in metrics_all.items():
        serializable[model] = []
        for fold in folds:
            fold_copy = {}
            for k, v in fold.items():
                if isinstance(v, np.ndarray):
                    fold_copy[k] = v.tolist()  # convert array to list
                elif isinstance(v, (np.float64, np.float32)):
                    fold_copy[k] = float(v)    # convert to float
                elif isinstance(v, (np.int64, np.int32)):
                    fold_copy[k] = int(v)      # convert to int
                else:
                    fold_copy[k] = v
            serializable[model].append(fold_copy)
    return serializable



# Model Training

### XGBoost initial run with k-fold

In [37]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', n_jobs=-1, random_state=42)
xgb_metrics = train_models_kfold(X, y, {"XGBoost": xgb_model}, k=8)

# Convert metrics to JSON-serializable format
xgb_metrics = make_json_serializable(xgb_metrics)
with open(os.path.join(RESULTS_DIR, "xgb_metrics.json"), "w") as f:
    json.dump(xgb_metrics, f, indent=4)

# Save the trained XGBoost model
# save_model(xgb_model, os.path.join(RESULTS_DIR, "xgb_initial_model.pkl"))
print("XGBoost initial run done. Metrics saved in xgb_metrics.json")


XGBoost initial run done. Metrics saved in xgb_metrics.json


### RandomForest initial run with k-fold

### Lightgbm initial run with k-fold

In [38]:
from lightgbm import LGBMClassifier

lgb_model = LGBMClassifier(n_jobs=-1, random_state=42)
lgb_metrics = train_models_kfold(X, y, {"LightGBM": lgb_model}, k=8)

# Convert metrics to JSON-serializable format
lgb_metrics = make_json_serializable(lgb_metrics)
with open(os.path.join(RESULTS_DIR, "lgb_metrics.json"), "w") as f:
    json.dump(lgb_metrics, f, indent=4)


print("LightGBM initial run done. Metrics saved in lgb_metrics.json")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.027947 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4727
[LightGBM] [Info] Number of data points in the train set: 547560, number of used features: 22
[LightGBM] [Info] Start training from score -2.357426
[LightGBM] [Info] Start training from score -2.872066
[LightGBM] [Info] Start training from score -2.429366
[LightGBM] [Info] Start training from score -2.417803
[LightGBM] [Info] Start training from score -1.642203
[LightGBM] [Info] Start training from score -1.224527
[LightGBM] [Info] Start training from score -2.749324
[LightGBM] [Info] Start training from score -3.340560
[LightGBM] [Info] Start training from score -2.468561
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.031305 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4724
[LightGBM] [Info] Number of 

In [39]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_jobs=-1, random_state=42)
rf_metrics = train_models_kfold(X, y, {"RandomForest": rf_model}, k=8)

# Convert metrics to JSON-serializable format
rf_metrics_serializable = make_json_serializable(rf_metrics)

# Save metrics
with open(os.path.join(RESULTS_DIR, "rf_metrics.json"), "w") as f:
    json.dump(rf_metrics_serializable, f, indent=4)

# Save the trained RandomForest model
# save_model(rf_model, os.path.join(RESULTS_DIR, "rf_initial_model.pkl"))

print("RandomForest initial run done. Metrics saved in rf_metrics.json")


RandomForest initial run done. Metrics saved in rf_metrics.json


In [11]:

# XGBoost hyperparameter tuning
best_xgb_params = optuna_tuning_xgb(X, y, n_trials=10)
print("Best XGB params:", best_xgb_params)

xgb_best = XGBClassifier(**best_xgb_params, use_label_encoder=False, eval_metric='mlogloss', n_jobs=-1, random_state=42)
xgb_best_metrics = train_models_kfold(X, y, {"XGBoost": xgb_best}, k=3)
xgb_best_metrics = make_json_serializable(xgb_best_metrics)
save_model(xgb_best, os.path.join(RESULTS_DIR, "xgb_best_model.pkl"))
with open(os.path.join(RESULTS_DIR, "xgb_best_params.json"), "w") as f:
    json.dump(best_xgb_params, f, indent=4)
with open(os.path.join(RESULTS_DIR, "xgb_best_metrics.json"), "w") as f:
    json.dump(xgb_best_metrics, f, indent=4)

print("XGBoost trained with best params. Model & metrics saved.")

[I 2025-11-28 13:13:32,292] A new study created in memory with name: no-name-e8720b0b-430c-4a74-a562-f0589a5a6fda
[I 2025-11-28 13:16:28,815] Trial 0 finished with value: 0.6667073956546394 and parameters: {'n_estimators': 429, 'max_depth': 3, 'learning_rate': 0.15165829961461974, 'subsample': 0.5704201610157282, 'colsample_bytree': 0.8225956950877971, 'gamma': 4.03403343314419}. Best is trial 0 with value: 0.6667073956546394.
[I 2025-11-28 13:19:40,609] Trial 1 finished with value: 0.665135495350003 and parameters: {'n_estimators': 421, 'max_depth': 15, 'learning_rate': 0.14377156727992166, 'subsample': 0.7781732624361821, 'colsample_bytree': 0.6771976192733182, 'gamma': 3.6475828034340574}. Best is trial 0 with value: 0.6667073956546394.
[I 2025-11-28 13:24:16,418] Trial 2 finished with value: 0.6629363373324044 and parameters: {'n_estimators': 496, 'max_depth': 3, 'learning_rate': 0.07835110688602241, 'subsample': 0.8640972640382156, 'colsample_bytree': 0.7199250805123616, 'gamma': 

Best XGB params: {'n_estimators': 498, 'max_depth': 4, 'learning_rate': 0.21884762175887876, 'subsample': 0.5107484744103502, 'colsample_bytree': 0.9175426877290372, 'gamma': 4.096948106787851}
XGBoost trained with best params. Model & metrics saved.


In [12]:
# RandomForest hyperparameter tuning
best_rf_params = optuna_tuning_rf(X, y, n_trials=20)
print("Best RF params:", best_rf_params)
rf_best = RandomForestClassifier(**best_rf_params, n_jobs=-1, random_state=42)
rf_best_metrics = train_models_kfold(X, y, {"RandomForest": rf_best}, k=5)
rf_best_metrics = make_json_serializable(rf_best_metrics)
save_model(rf_best, os.path.join(RESULTS_DIR, "rf_best_model.pkl"))
with open(os.path.join(RESULTS_DIR, "rf_best_params.json"), "w") as f:
    json.dump(best_rf_params, f, indent=4)
with open(os.path.join(RESULTS_DIR, "rf_best_metrics.json"), "w") as f:
    json.dump(rf_best_metrics, f, indent=4)
print("RandomForest trained with best params. Model & metrics saved.")


[I 2025-11-28 13:53:25,422] A new study created in memory with name: no-name-44d5340f-e4d9-4d94-8579-0cc3ba2cc711
[I 2025-11-28 13:55:35,301] Trial 0 finished with value: 0.5049593278793921 and parameters: {'n_estimators': 440, 'max_depth': 5, 'min_samples_split': 9, 'min_samples_leaf': 4}. Best is trial 0 with value: 0.5049593278793921.
[I 2025-11-28 13:58:18,213] Trial 1 finished with value: 0.6428387743610667 and parameters: {'n_estimators': 265, 'max_depth': 29, 'min_samples_split': 10, 'min_samples_leaf': 5}. Best is trial 1 with value: 0.6428387743610667.
[I 2025-11-28 14:03:00,618] Trial 2 finished with value: 0.642414851968602 and parameters: {'n_estimators': 473, 'max_depth': 23, 'min_samples_split': 9, 'min_samples_leaf': 2}. Best is trial 1 with value: 0.6428387743610667.
[I 2025-11-28 14:04:40,785] Trial 3 finished with value: 0.5343223079648562 and parameters: {'n_estimators': 262, 'max_depth': 7, 'min_samples_split': 6, 'min_samples_leaf': 5}. Best is trial 1 with value: 

Best RF params: {'n_estimators': 154, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 1}
RandomForest trained with best params. Model & metrics saved.


## LightGBM hyperparameter tuning

In [13]:

best_lgb_params = optuna_tuning_lgb(X, y, n_trials=10)
print("Best LGB params:", best_lgb_params)


[I 2025-11-28 15:03:56,333] A new study created in memory with name: no-name-02d94812-6cbb-4abc-8fb5-5dcdc436da56


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.025607 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4720
[LightGBM] [Info] Number of data points in the train set: 417188, number of used features: 22
[LightGBM] [Info] Start training from score -2.360203
[LightGBM] [Info] Start training from score -2.864230
[LightGBM] [Info] Start training from score -2.428991
[LightGBM] [Info] Start training from score -2.417578
[LightGBM] [Info] Start training from score -1.641350
[LightGBM] [Info] Start training from score -1.225377
[LightGBM] [Info] Start training from score -2.747601
[LightGBM] [Info] Start training from score -3.340939
[LightGBM] [Info] Start training from score -2.471497
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

[I 2025-11-28 15:07:50,822] Trial 0 finished with value: 0.6570387734964803 and parameters: {'n_estimators': 273, 'max_depth': 9, 'learning_rate': 0.05640859489576205, 'num_leaves': 125, 'subsample': 0.7903083947435792, 'colsample_bytree': 0.5819620469125051}. Best is trial 0 with value: 0.6570387734964803.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.031429 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4720
[LightGBM] [Info] Number of data points in the train set: 417188, number of used features: 22
[LightGBM] [Info] Start training from score -2.360203
[LightGBM] [Info] Start training from score -2.864230
[LightGBM] [Info] Start training from score -2.428991
[LightGBM] [Info] Start training from score -2.417578
[LightGBM] [Info] Start training from score -1.641350
[LightGBM] [Info] Start training from score -1.225377
[LightGBM] [Info] Start training from score -2.747601
[LightGBM] [Info] Start training from score -3.340939
[LightGBM] [Info] Start training from score -2.471497
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

[I 2025-11-28 15:12:42,585] Trial 1 finished with value: 0.6504615480583588 and parameters: {'n_estimators': 334, 'max_depth': 8, 'learning_rate': 0.12134212279373267, 'num_leaves': 100, 'subsample': 0.5534096268186272, 'colsample_bytree': 0.5981183320221257}. Best is trial 0 with value: 0.6570387734964803.


...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.120153 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4720
[LightGBM] [Info] Number of data points in the train set: 417188, number of used features: 22
[LightGBM] [Info] Start training from score -2.360203
[LightGBM] [Info] Start training from score -2.864230
[LightGBM] [Info] Start training from score -2.428991
[LightGBM] [Info] Start training from score -2.417578
[LightGBM] [Info] Start training from score -1.641350
[LightGBM] [Info] Start training from score -1.225377
[LightGBM] [Info] Start training from score -2.747601
[LightGBM] [Info] Start training from score -3.340939
[LightGBM] [Info] Start training from score -2.471497
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

[I 2025-11-28 15:55:25,940] Trial 8 finished with value: 0.5414459903721521 and parameters: {'n_estimators': 415, 'max_depth': 14, 'learning_rate': 0.26408841409022515, 'num_leaves': 57, 'subsample': 0.9023426399721973, 'colsample_bytree': 0.8193296471535958}. Best is trial 3 with value: 0.6712636271460418.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012624 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4720
[LightGBM] [Info] Number of data points in the train set: 417188, number of used features: 22
[LightGBM] [Info] Start training from score -2.360203
[LightGBM] [Info] Start training from score -2.864230
[LightGBM] [Info] Start training from score -2.428991
[LightGBM] [Info] Start training from score -2.417578
[LightGBM] [Info] Start training from score -1.641350
[LightGBM] [Info] Start training from score -1.225377
[LightGBM] [Info] Start training from score -2.747601
[LightGBM] [Info] Start training from score -3.340939
[LightGBM] [Info] Start training from score -2.471497
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM]

[I 2025-11-28 16:05:57,554] Trial 9 finished with value: 0.6505656856366878 and parameters: {'n_estimators': 347, 'max_depth': 14, 'learning_rate': 0.07409560032430985, 'num_leaves': 103, 'subsample': 0.804459323709177, 'colsample_bytree': 0.545368879010024}. Best is trial 3 with value: 0.6712636271460418.


Best LGB params: {'n_estimators': 180, 'max_depth': 3, 'learning_rate': 0.1916837531298268, 'num_leaves': 143, 'subsample': 0.7978695746237181, 'colsample_bytree': 0.5957037951509012}


In [14]:
# Build final LightGBM model
lgb_best = LGBMClassifier(**best_lgb_params, n_jobs=-1, random_state=42)

# Train with K-Fold
lgb_best_metrics = train_models_kfold(X, y, {"LightGBM": lgb_best}, k=3)

# Convert metrics to JSON-safe format
lgb_best_metrics = make_json_serializable(lgb_best_metrics)

# Save model
save_model(lgb_best, os.path.join(RESULTS_DIR, "lgb_best_model.pkl"))

# Save best params
with open(os.path.join(RESULTS_DIR, "lgb_best_params.json"), "w") as f:
    json.dump(best_lgb_params, f, indent=4)

# Save metrics
with open(os.path.join(RESULTS_DIR, "lgb_best_metrics.json"), "w") as f:
    json.dump(lgb_best_metrics, f, indent=4)

print("LightGBM trained with best params. Model & metrics saved.")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.052400 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4722
[LightGBM] [Info] Number of data points in the train set: 417188, number of used features: 22
[LightGBM] [Info] Start training from score -2.360203
[LightGBM] [Info] Start training from score -2.864230
[LightGBM] [Info] Start training from score -2.428991
[LightGBM] [Info] Start training from score -2.417578
[LightGBM] [Info] Start training from score -1.641350
[LightGBM] [Info] Start training from score -1.225377
[LightGBM] [Info] Start training from score -2.747601
[LightGBM] [Info] Start training from score -3.340939
[LightGBM] [Info] Start training from score -2.471497
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

NameError: name 'train_models_kfold' is not defined